# exercise_2.2.1_Data Types and Subsetting

This exercise uses `datania_households_raw.csv` and focuses on data types, type-safe loading, anomaly diagnostics, and targeted subsetting.

### Path Setup (run first)

> Use `os.path.join` for path construction.
> Required base path: `DATA_RAW_DIR = "../../data/0_raw"`.

In [ ]:
import os
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1 — Why loading as string matters (leading zeros)

Load the same file twice: first with default type inference, then with explicit string dtypes for IDs/codes.

In [ ]:
# Default loading (can drop leading zeros in codes)
df_default = pd.read_csv(raw_path)
print(df_default[['hh_id', 'region_code']].head(8))
df_default.dtypes

In [ ]:
# Type-safe loading for identifier-like columns
df = pd.read_csv(
    raw_path,
    dtype={'hh_id': 'string', 'region_code': 'string'}
)
print(df[['hh_id', 'region_code']].head(8))
df.dtypes

**Questions:**

- What changed between `df_default` and `df` for `region_code`?
- Why should `hh_id` and `region_code` be treated as text rather than numeric?
- Which future tasks (joins/merges/grouping) could fail if leading zeros are lost?

---

## Task 2 — DataFrame vs Series quick check

In [ ]:
print(type(df))
print(type(df['income_dkw']))
print(df['income_dkw'].dtype)

**Question:** Which operations below are Series-level operations: `.dtype`, `.str`, `.dt`, `.unique()`, `.value_counts()`?

---

## Task 3 — Diagnose structure and anomalies with `info`, `count`, and `describe`

In [ ]:
df.info(verbose=True)

In [ ]:
# Non-missing count per column
df.count()

In [ ]:
df.describe(include="all").T

**Questions:**

- Which columns clearly have mixed formats?
- Which fields show impossible/extreme values (`hh_size`, `age`, `pop_density`)?
- Which columns need conversion before valid analysis?

---

## Task 4 — Use `unique()` and `value_counts()` to inspect categories

In [ ]:
df['urban_rural'].unique()

In [ ]:
df['region_code'].unique()

In [ ]:
df['province_name'].value_counts(dropna=False)

**Questions:**

- Do you find typo categories (example: `Urbn`)?
- Are missing categories visible when `dropna=False` is used?
- Is `region_code` standardized (e.g., `01` vs `1` vs empty)?

---

## Task 5 — Clean `income_dkw` (main focus)

The raw values include symbols, spaces, commas, text labels, and missing markers.

In [ ]:
df['income_dkw'].head(15)

In [ ]:
# Keep original column, then create a cleaned numeric version
income_text = df['income_dkw'].astype('string')

# TODO (student): complete the cleaning pipeline below
# Goal: remove 'Ar', commas, spaces, and surrounding whitespace
df['income_dkw_clean'] = (
    income_text
    # .str.replace('Ar', '', regex=False)
    # .str.replace(',', '', regex=False)
    # .str.replace(' ', '', regex=False)
    # .str.strip()
)

# TODO (student): convert cleaned text to numeric
df['income_dkw_num'] = pd.to_numeric(df['income_dkw_clean'], errors='coerce')

df[['income_dkw', 'income_dkw_clean', 'income_dkw_num']].head(20)

In [ ]:
df['income_dkw_num'].describe()

In [ ]:
# Rows where conversion failed
df[df['income_dkw_num'].isna()][['hh_id', 'income_dkw']]

In [ ]:
# Potentially invalid/odd values after conversion
df[df['income_dkw_num'] < 0][['hh_id', 'income_dkw', 'income_dkw_num']]

**Questions:**

- How many values became `NaN` after numeric conversion, and why?
- Which records contain impossible income values?
- Should very high incomes be treated as errors, outliers, or valid extremes?

---

## Task 6 — Convert `survey_date` and inspect failures

In [ ]:
df['survey_date_parsed'] = pd.to_datetime(df['survey_date'], errors='coerce')
df[['survey_date', 'survey_date_parsed']].head(15)

In [ ]:
df[df['survey_date_parsed'].isna()][['hh_id', 'survey_date']]

In [ ]:
df['survey_month'] = df['survey_date_parsed'].dt.month
df['survey_month'].value_counts(dropna=False).sort_index()

---

## Task 7 — Subsetting for diagnostics (do not drop yet)

In [ ]:
hh_size_num = pd.to_numeric(df['hh_size'], errors='coerce')
age_num = pd.to_numeric(df['age'], errors='coerce')

invalid_size = df[(hh_size_num <= 0) | (hh_size_num > 20)]
invalid_age = df[(age_num < 0) | (age_num > 120)]
duplicates = df[df['hh_id'].duplicated(keep=False)]
bad_settlement = df[~df['urban_rural'].isin(['Urban', 'Rural'])]

print('Invalid hh_size rows:', len(invalid_size))
print('Invalid age rows:', len(invalid_age))
print('Duplicate hh_id rows:', len(duplicates))
print('Unexpected urban_rural rows:', len(bad_settlement))

In [ ]:
invalid_size[['hh_id', 'hh_size']]

In [ ]:
duplicates[['hh_id', 'survey_date', 'income_dkw']]

**Wrap-up questions:**

- Which 5 data issues are highest priority before analysis?
- Which fields should remain text permanently (IDs/codes) and why?
- Which cleaning rules would you codify in a reusable pipeline?